In [ ]:
!pip uninstall -y kaggle kagglesdk kagglehub

In [ ]:
!pip install -U kagglesdk
!pip install -U kaggle
!pip install -U kagglehub

In [ ]:
!pip install -U transformers datasets accelerate peft bitsandbytes huggingface_hub

In [4]:
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [5]:
import kagglehub

In [6]:
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [7]:
from datasets import load_dataset
ds = load_dataset("databricks/databricks-dolly-15k")

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})

In [9]:
train_ds = ds["train"]
train_ds

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})

In [10]:
data = []

for example in train_ds:
    # Skip rows where context is not empty
    if example["context"]:
        continue

    template = "Instruction:\n{instruction}\n\nResponse:\n{response}"
    text = template.format(
        instruction=example["instruction"],
        response=example["response"]
    )

    data.append(text)
    # Take first 1000 samples
    if len(data) == 1000:
        break

print(len(data))
print(data[0])

1000
Instruction:
Which is a species of fish? Tope or Rope

Response:
Tope


In [11]:
from datasets import Dataset

dataset = Dataset.from_dict({"text": data})

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

#Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

#Since we're using free GPU. Thus quantized the base model to 4-bit as free GPU
# - memory is not enough for full scale model loading.
#Thus we are officially now doing QLoRA finetuning.
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [14]:
from peft import prepare_model_for_kbit_training
frozen_model = prepare_model_for_kbit_training(model)

In [15]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=4,                         # LoRA rank (your old rank=4)
    lora_alpha=16,               # scaling factor
    target_modules=["q_proj", "v_proj"],  # very important
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [16]:
lora_model = get_peft_model(frozen_model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 460,800 || all params: 2,506,633,216 || trainable%: 0.0184


In [17]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    # eval_strategy="steps",   # NEW
    # eval_steps=50,                 # Evaluate every 50 steps
    save_strategy="epoch",
    fp16=True
)

1️⃣ output_dir="./results"

Where model checkpoints and logs are saved.

After training you’ll see:

./results/
    checkpoint-xxx/
    
<br/>2️⃣ per_device_train_batch_size=2

Batch size per GPU.

If:

You have 1 GPU → total batch size = 2

You have 2 GPUs → total batch size = 4

<br/>3️⃣ gradient_accumulation_steps=4

This simulates a larger batch size.

Effective batch size:

per_device_train_batch_size × gradient_accumulation_steps
= 2 × 4 = 8

Instead of updating weights every 2 samples,
it accumulates gradients for 4 steps, then updates.

Very useful when GPU memory is small.

<br/>4️⃣ num_train_epochs=1

How many times the full dataset is passed.

1 epoch = full dataset once.

<br/>5️⃣ logging_steps=10

Logs training loss every 10 optimizer steps.

This controls how often you see logs.

<br/>6️⃣ save_strategy="epoch"

Model checkpoint saving strategy:

Options:

"epoch" → save after each epoch

"steps" → save every save_steps

"no" → don’t save

<br/>7️⃣ fp16=True

Use half precision (16-bit floats).

Reduces memory.
Speeds up training.
Works only if GPU supports it.

For QLoRA this is usually safe.

In [18]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_ds = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_ds,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,21.250307
20,20.675572
30,21.167766
40,20.428738
50,21.842691
60,20.576303
70,20.585332
80,20.757457
90,19.898642
100,20.593997


TrainOutput(global_step=125, training_loss=20.787235595703127, metrics={'train_runtime': 645.9394, 'train_samples_per_second': 1.548, 'train_steps_per_second': 0.194, 'total_flos': 6089764503552000.0, 'train_loss': 20.787235595703127, 'epoch': 1.0})

In [23]:
trainer.model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): Embedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [22]:
finetune_model = trainer.model
finetune_model.eval()

prompt = "Explain what is photosynthesis."

inputs = tokenizer(prompt, return_tensors="pt").to(finetune_model.device)

with torch.no_grad():
    outputs = finetune_model.generate(
        **inputs,
        max_new_tokens=200,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Explain what is photosynthesis.

The following is a crude but effective method for estimating the approximate mass of an object: Drop a spherical object and a cylindrical object made of the same material and using the same initial drop conditions into a body of water. If the spheres have the same diameter and are made of the same material and the cylindrical object is $90 \%$ of the diameter of the sphere, will it drop faster than the sphere? Explain.

A 1000-W iron is left on an ironing board with the iron idling. If the coefficient of kinetic friction between the iron and the board is $0.25,$ how much mass must be removed from the iron to prevent it from moving when the iron is switched on?

A 1000-W iron is left on an ironing board with the iron idling. If the coefficient of kinetic friction between the iron and the board is 0.25, how much mass must be removed from the iron to prevent it from moving when
